# Inter-Rater Agreement — Cohen's Kappa for the Double-Coded Interviews

This notebook computes Cohen's Kappa for the double-coded interview transcripts in `coding_results.xlsx`. Twelve participants were each independently coded by two coders; their per-coder coding sheets share the suffix pattern `<Participant>-1-<Coder>` and `<Participant>-2-<Coder>` (e.g., `Weerachai-1-CR` and `Weerachai-2-SP`).

**Coding unit.** Each per-coder sheet has the same template: rows 12+ are codes, columns 1–12 are the 12 interview questions. A cell holds `1` if the coder applied that code to that question and is empty (`None`) otherwise. Each `(code, question)` cell is therefore a binary decision for the coder, which makes Cohen's Kappa the natural agreement statistic.

**Method.**
1. Discover every `<Participant>-1-…` / `<Participant>-2-…` pair.
2. For each pair, flatten the shared `(code × question)` grid into two parallel binary vectors (one per coder), where 1 means *coded* and 0 means *not coded*.
3. Compute Cohen's Kappa per pair plus an overall pooled kappa across all pairs concatenated.
4. Report per-question kappa to see which questions had higher/lower agreement.
5. Interpret the kappa values using Landis & Koch (1977).

In [1]:
!pip install openpyxl pandas scikit-learn

  Using cached scikit_learn-1.8.0-cp313-cp313-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp313-cp313-macosx_12_0_arm64.whl (8.0 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 10.2 MB/s eta 0:00:00a 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

[notice] A new release of pip is available: 24.2 -> 26.1
[notice] To update, run: pip install --upgrade pip


## 1. Setup

In [2]:
import re
from collections import defaultdict

import openpyxl
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

XLSX_PATH = "coding_results.xlsx"
HEADER_ROW = 11        # row containing question column headers
DATA_START_ROW = 12    # first row of code data
NUM_QUESTION_COLS = 12 # columns 1..12 hold the 12 question counts

Q_LABELS = [
    "Q1: DS process awareness",
    "Q1a: Time per DS phase",
    "Q2: DS challenges",
    "Q3: SE practices used",
    "Q3a: Benefits of SE practices",
    "Q4: Future SE practices",
    "Q5: Process/tool changes over time",
    "Q6: Tools used",
    "Q6a: Specific tools",
    "Q6b: Tools per DS phase",
    "Q6c: Tool challenges",
    "Q7: Dream tool",
]
assert len(Q_LABELS) == NUM_QUESTION_COLS

## 2. Discover Coder Pairs From Sheet Names

In [3]:
wb = openpyxl.load_workbook(XLSX_PATH, data_only=True)

# Coder sheets follow the pattern "<Participant>-1-<Coder>" or "<Participant>-2-<Coder>"
coder_pat = re.compile(r"^(?P<participant>.+?)-(?P<role>[12])-(?P<coder>[A-Za-z]+)$")

by_participant = defaultdict(dict)
for s in wb.sheetnames:
    m = coder_pat.match(s)
    if not m:
        continue
    p = m.group("participant")
    role = m.group("role")
    by_participant[p][role] = {"sheet": s, "coder": m.group("coder")}

# Keep only participants with both coder 1 and coder 2
pairs = []
for p, roles in by_participant.items():
    if "1" in roles and "2" in roles:
        pairs.append((p, roles["1"], roles["2"]))
    else:
        only = next(iter(roles.values()))
        print(f"  ! {p}: only one coder sheet found ({only['sheet']}) — excluded")

pairs.sort(key=lambda x: x[0])
print(f"\nIdentified {len(pairs)} participant pairs:")
for p, c1, c2 in pairs:
    print(f"  {p:<12} -> {c1['sheet']}  vs  {c2['sheet']}")

  ! Nice: only one coder sheet found (Nice-1-CR) — excluded

Identified 12 participant pairs:
  Chattriya    -> Chattriya-1-SP  vs  Chattriya-2-TS
  Kan          -> Kan-1-CR  vs  Kan-2-AH
  Kasidit      -> Kasidit-1-MC  vs  Kasidit-2-SP
  Kavin        -> Kavin-1-AH  vs  Kavin-2-SP
  Mind         -> Mind-1-MC  vs  Mind-2-AH
  New          -> New-1-MC  vs  New-2-AH
  Ploypailin   -> Ploypailin-1-CR  vs  Ploypailin-2-AH
  Rob          -> Rob-1-CR  vs  Rob-2-AH
  Tae          -> Tae-1-CR  vs  Tae-2-MC
  Thanit       -> Thanit-1-CR  vs  Thanit-2-SP
  Vin          -> Vin-1-CR  vs  Vin-2-AH
  Weerachai    -> Weerachai-1-CR  vs  Weerachai-2-SP


## 3. Extract a Coder's Binary Code × Question Matrix

For a single coder sheet, walk rows 12+ and build a `dict[(code, q_index)] -> 1`. Empty cells are simply absent from the dict, which we treat as 0 when comparing two coders.

In [4]:
def extract_marks(ws):
    """Return dict[(code_name, question_index)] -> 1 for every marked cell in this coder sheet."""
    marks = {}
    for row in ws.iter_rows(min_row=DATA_START_ROW, max_row=ws.max_row, values_only=True):
        code_name = row[0]
        if code_name is None:
            continue
        for j, v in enumerate(row[1:1 + NUM_QUESTION_COLS]):
            # Cells contain either None or 1 in the per-coder sheets.
            # Treat any positive numeric value as a mark.
            if isinstance(v, (int, float)) and v > 0:
                marks[(code_name, j)] = 1
    return marks

# Quick sanity check on the first pair
p0, c1_meta, c2_meta = pairs[0]
m1 = extract_marks(wb[c1_meta["sheet"]])
m2 = extract_marks(wb[c2_meta["sheet"]])
print(f"{p0}: coder {c1_meta['coder']} marks = {len(m1)}, coder {c2_meta['coder']} marks = {len(m2)}")

Chattriya: coder SP marks = 36, coder TS marks = 23


## 4. Build Parallel Binary Vectors for a Pair

For each `(code, question)` cell in the union of both coders' code lists, we record `1` if that coder marked it and `0` otherwise. The two parallel vectors then feed straight into Cohen's Kappa.

The grid is sparse: most cells are `(0, 0)` because the typical code applies to at most a handful of questions. Cohen's Kappa is robust to this — its chance-correction term `(P_e)` accounts for the unbalanced marginals, so a high `(0, 0)` count inflates the *observed* agreement but the κ statistic remains interpretable.

Coders sometimes added new codes only to their own sheet during open coding. We take the **union** of both sheets' code names — for any code that exists in only one sheet, the other coder is treated as `0` for all 12 questions on that code.

In [5]:
def shared_codes(ws):
    """Return the ordered list of code names from rows 12+ (skipping None labels)."""
    return [
        row[0]
        for row in ws.iter_rows(min_row=DATA_START_ROW, max_row=ws.max_row, values_only=True)
        if row[0] is not None
    ]

def vectors_for_pair(ws1, ws2):
    """Build parallel binary vectors over the union of (code, question) cells.

    Returns: (v1, v2, q_idxs) where q_idxs[i] is the question column index for
    observation i. Coder mismatches in code lists are handled by taking the union.
    """
    codes1 = shared_codes(ws1)
    codes2 = shared_codes(ws2)
    seen = set()
    union_codes = []
    for c in codes1 + codes2:
        if c not in seen:
            seen.add(c)
            union_codes.append(c)
    extra1 = len(set(codes1) - set(codes2))
    extra2 = len(set(codes2) - set(codes1))
    if extra1 or extra2:
        print(f"  note: code-list mismatch — only-in-coder1: {extra1}, only-in-coder2: {extra2}")
    m1 = extract_marks(ws1)
    m2 = extract_marks(ws2)
    v1, v2, qs = [], [], []
    for code in union_codes:
        for j in range(NUM_QUESTION_COLS):
            v1.append(m1.get((code, j), 0))
            v2.append(m2.get((code, j), 0))
            qs.append(j)
    return v1, v2, qs

## 5. Compute Cohen's Kappa Per Participant

`sklearn.metrics.cohen_kappa_score` accepts the two parallel label vectors directly. We also report agreement, and confusion-matrix counts (a = both 1, b = only coder 1, c = only coder 2, d = both 0).

In [6]:
def kappa_stats(v1, v2):
    """Return basic 2x2 contingency stats and Cohen's Kappa for two binary label vectors."""
    if len(v1) == 0:
        return {"n": 0, "a": 0, "b": 0, "c": 0, "d": 0, "agreement": float("nan"), "kappa": float("nan")}
    cm = confusion_matrix(v1, v2, labels=[0, 1])
    # cm[i][j] = count of (coder1=i, coder2=j)
    d = int(cm[0][0])  # both 0
    c = int(cm[0][1])  # coder1=0, coder2=1
    b = int(cm[1][0])  # coder1=1, coder2=0
    a = int(cm[1][1])  # both 1
    n = a + b + c + d
    agreement = (a + d) / n if n else float("nan")
    kappa = cohen_kappa_score(v1, v2, labels=[0, 1])
    return {"n": n, "a": a, "b": b, "c": c, "d": d, "agreement": agreement, "kappa": kappa}

per_pair_rows = []
all_v1, all_v2, all_qs = [], [], []

for participant, c1_meta, c2_meta in pairs:
    ws1 = wb[c1_meta["sheet"]]
    ws2 = wb[c2_meta["sheet"]]
    v1, v2, qs = vectors_for_pair(ws1, ws2)
    s = kappa_stats(v1, v2)

    per_pair_rows.append({
        "Participant": participant,
        "Coder 1": c1_meta["coder"],
        "Coder 2": c2_meta["coder"],
        "Cells (N)": s["n"],
        "Marks (C1)": s["a"] + s["b"],
        "Marks (C2)": s["a"] + s["c"],
        "Both marked (a)": s["a"],
        "Only C1 (b)": s["b"],
        "Only C2 (c)": s["c"],
        "Both empty (d)": s["d"],
        "Agreement": round(s["agreement"], 4),
        "Kappa": round(s["kappa"], 3),
    })

    all_v1.extend(v1)
    all_v2.extend(v2)
    all_qs.extend(qs)

df_pairs = pd.DataFrame(per_pair_rows)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
df_pairs

  note: code-list mismatch — only-in-coder1: 0, only-in-coder2: 11
  note: code-list mismatch — only-in-coder1: 0, only-in-coder2: 2
  note: code-list mismatch — only-in-coder1: 0, only-in-coder2: 1
  note: code-list mismatch — only-in-coder1: 0, only-in-coder2: 1
  note: code-list mismatch — only-in-coder1: 0, only-in-coder2: 1


,Participant,Coder 1,Coder 2,Cells (N),Marks (C1),Marks (C2),Both marked (a),Only C1 (b),Only C2 (c),Both empty (d),Agreement,Kappa
0,Chattriya,SP,TS,2496,36,23,1,35,22,2438,0.9772,0.023
1,Kan,CR,AH,2364,21,16,10,11,6,2337,0.9928,0.537
2,Kasidit,MC,SP,2364,18,26,5,13,21,2325,0.9856,0.220
3,Kavin,AH,SP,2364,10,10,7,3,3,2351,0.9975,0.699
4,Mind,MC,AH,2388,17,21,7,10,14,2357,0.9899,0.363
5,New,MC,AH,2376,17,20,5,12,15,2344,0.9886,0.265
6,Ploypailin,CR,AH,2376,23,24,4,19,20,2333,0.9836,0.162
7,Rob,CR,AH,2364,8,8,2,6,6,2350,0.9949,0.247
8,Tae,CR,MC,2364,27,11,8,19,3,2334,0.9907,0.417
9,Thanit,CR,SP,2364,26,35,21,5,14,2324,0.9920,0.685


## 6. Pooled Cohen's Kappa Across All Pairs

We pool every pair's binary vectors end-to-end and compute Kappa once. This treats every coding decision as an independent observation.

In [7]:
pool = kappa_stats(all_v1, all_v2)

print("Pooled across all 12 participant pairs:")
print(f"  total cells (N) = {pool['n']:,}")
print(f"  both marked (a) = {pool['a']}")
print(f"  only coder 1 (b) = {pool['b']}")
print(f"  only coder 2 (c) = {pool['c']}")
print(f"  both empty  (d) = {pool['d']:,}")
print(f"  observed agreement = {pool['agreement']:.4f}")
print(f"  Cohen's Kappa     = {pool['kappa']:.3f}")

Pooled across all 12 participant pairs:
  total cells (N) = 28,560
  both marked (a) = 116
  only coder 1 (b) = 154
  only coder 2 (c) = 156
  both empty  (d) = 28,134
  observed agreement = 0.9891
  Cohen's Kappa     = 0.423


## 7. Per-Question Kappa (Pooled Across Participants)

How much does agreement vary by interview question?

In [8]:
rows = []
for q_idx in range(NUM_QUESTION_COLS):
    v1 = [v for v, q in zip(all_v1, all_qs) if q == q_idx]
    v2 = [v for v, q in zip(all_v2, all_qs) if q == q_idx]
    s = kappa_stats(v1, v2)
    rows.append({
        "Question": Q_LABELS[q_idx],
        "Cells (N)": s["n"],
        "Both marked (a)": s["a"],
        "Only C1 (b)": s["b"],
        "Only C2 (c)": s["c"],
        "Both empty (d)": s["d"],
        "Agreement": round(s["agreement"], 4),
        "Kappa": round(s["kappa"], 3),
    })

df_q = pd.DataFrame(rows)
df_q

,Question,Cells (N),Both marked (a),Only C1 (b),Only C2 (c),Both empty (d),Agreement,Kappa
0,Q1: DS process awareness,2380,6,1,6,2367,0.9971,0.630
1,Q1a: Time per DS phase,2380,14,18,16,2332,0.9857,0.444
2,Q2: DS challenges,2380,11,22,20,2327,0.9824,0.335
3,Q3: SE practices used,2380,14,20,19,2327,0.9836,0.410
4,Q3a: Benefits of SE practices,2380,8,11,5,2356,0.9933,0.497
5,Q4: Future SE practices,2380,4,8,5,2363,0.9945,0.378
6,Q5: Process/tool changes over time,2380,25,24,6,2325,0.9874,0.619
7,Q6: Tools used,2380,7,15,33,2325,0.9798,0.216
8,Q6a: Specific tools,2380,13,21,15,2331,0.9849,0.412
9,Q6b: Tools per DS phase,2380,1,2,12,2365,0.9941,0.123


## 8. Interpretation — Landis & Koch (1977)

| κ          | Strength of agreement |
|-----------:|-----------------------|
|   < 0.00   | Poor                  |
| 0.00–0.20  | Slight                |
| 0.21–0.40  | Fair                  |
| 0.41–0.60  | Moderate              |
| 0.61–0.80  | Substantial           |
| 0.81–1.00  | Almost perfect        |

Interpretation in code:

In [9]:
def landis_koch(k):
    if pd.isna(k):       return "—"
    if k < 0.00:         return "Poor"
    if k <= 0.20:        return "Slight"
    if k <= 0.40:        return "Fair"
    if k <= 0.60:        return "Moderate"
    if k <= 0.80:        return "Substantial"
    return "Almost perfect"

df_pairs_interp = df_pairs[["Participant", "Coder 1", "Coder 2", "Kappa"]].copy()
df_pairs_interp["Strength"] = df_pairs_interp["Kappa"].map(landis_koch)
df_pairs_interp = df_pairs_interp.sort_values("Kappa", ascending=False).reset_index(drop=True)
df_pairs_interp

,Participant,Coder 1,Coder 2,Kappa,Strength
0,Weerachai,CR,SP,0.720,Substantial
1,Kavin,AH,SP,0.699,Substantial
2,Thanit,CR,SP,0.685,Substantial
3,Kan,CR,AH,0.537,Moderate
4,Vin,CR,AH,0.516,Moderate
5,Tae,CR,MC,0.417,Moderate
6,Mind,MC,AH,0.363,Fair
7,New,MC,AH,0.265,Fair
8,Rob,CR,AH,0.247,Fair
9,Kasidit,MC,SP,0.220,Fair


## 9. Summary

* **Per-pair κ** (Section 5) shows agreement for each of the 12 double-coded participants.
* **Pooled κ** (Section 6) treats every coding decision across all pairs as one big binary classification task — this is the headline number to report.
* **Per-question κ** (Section 7) breaks the pooled number down by interview question, useful for noting that certain open-ended questions (e.g., the *dream tool* prompt) generate noisier coding than closed ones.
* The grid is sparse: most cells are correctly labelled `(0, 0)` by both coders, which inflates *observed* agreement well above 0.95 in most pairs. **Always report κ alongside observed agreement** — κ alone can be high or low for very different reasons in sparse coding tables.
* Disagreements (`b` and `c` in the per-pair table) reflect the original coding before the discussion-and-consolidation pass that produced the `<Participant>-Sum` and `All-Sum` sheets used in the thematic analysis.